In [ ]:
from pathlib import Path
import os
import pandas as pd
import re
from datetime import datetime
from pathlib import Path

# For defining Instrument
import requests
import pandas as pd

from decimal import Decimal
from pathlib import Path


from nautilus_trader.model.identifiers import InstrumentId, Symbol
from nautilus_trader.model.instruments import Instrument, CryptoPerpetual
from nautilus_trader.model.objects import Currency, Price, Quantity, Money
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from nautilus_trader.model.data import Bar
from nautilus_trader.model.data import BarType
from nautilus_trader.persistence.wranglers_v2 import BarDataWranglerV2


In [ ]:
### Use the below for regular kline: ###
PREMIUM_KLINE = False

if not PREMIUM_KLINE:
    SYMBOL = "ETHUSDT"
    EXCHANGE = "BYBIT"
    INSTRUMENT_ID = f"{SYMBOL}-LINEAR.{EXCHANGE}"
    instrument_id = InstrumentId.from_str(INSTRUMENT_ID)
    DATA_DIR = Path(os.environ.get("DATA_DIR", "~/desktop/tmpMarketData/KlineData/kline")).expanduser() / SYMBOL / "interval=1"
    CATALOG_DIR = Path(os.getcwd()).parent/"nautilusDataCatalog"
else:
    ### Use the below for premium kline: ###
    SYMBOL = "ETHUSDT"
    EXCHANGE = "BYBIT"

    # Standard instrument ID formulation
    INSTRUMENT_ID = f"{SYMBOL}-LINEAR.{EXCHANGE}"
    instrument_id = InstrumentId.from_str(INSTRUMENT_ID)

    # 1. Point the raw data directory to the new premium kline folder
    DATA_DIR = Path(os.environ.get("DATA_DIR", "~/desktop/tmpMarketData/KlineData/klinePremium")).expanduser() / SYMBOL / "interval=1"

    # 2. Point the catalog to a NEW isolated premium catalog directory
    CATALOG_DIR = Path(os.getcwd()).parent / "nautilusPremiumCatalog"

    # Ensure the new catalog directory actually exists before Nautilus tries to write to it
    CATALOG_DIR.mkdir(parents=True, exist_ok=True)



In [38]:
path = DATA_DIR

# 1. Compile the filename date-range pattern once
DATE_REGEX = re.compile(
    r"_(\d{4}-\d{2}-\d{2})_(\d{4}-\d{2}-\d{2})\.csv$"
)

def extract_file_dates(file_path: Path) -> tuple[datetime.date, datetime.date]:
    """
    Extract the start and end dates from filenames such as:

    ETHUSDT_kline_1_2021-04-01_2021-04-30.csv
    """
    match = DATE_REGEX.search(file_path.name)

    if not match:
        raise ValueError(
            f"Pipeline Stop: Filename '{file_path.name}' does not match "
            "'<symbol>_kline_<interval>_YYYY-MM-DD_YYYY-MM-DD.csv'."
        )

    start_date = datetime.strptime(match.group(1), "%Y-%m-%d").date()
    end_date = datetime.strptime(match.group(2), "%Y-%m-%d").date()

    if end_date < start_date:
        raise ValueError(
            f"Pipeline Stop: End date precedes start date in '{file_path.name}'."
        )

    return start_date, end_date


# 2. Sort chronologically by start date, then end date
raw_files = sorted(
    [
        f
        for f in path.iterdir()
        if f.is_file() and f.name.endswith(".csv")
    ],
    key=extract_file_dates,
)

assert raw_files, f"Unable to find any CSV files in directory {path}"

raw_files

[PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/KlineData/klinePremium/ETHUSDT/interval=1/ETHUSDT_premium_index_kline_1_2021-03-01_2021-03-31.csv'),
 PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/KlineData/klinePremium/ETHUSDT/interval=1/ETHUSDT_premium_index_kline_1_2021-04-01_2021-04-30.csv'),
 PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/KlineData/klinePremium/ETHUSDT/interval=1/ETHUSDT_premium_index_kline_1_2021-05-01_2021-05-31.csv'),
 PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/KlineData/klinePremium/ETHUSDT/interval=1/ETHUSDT_premium_index_kline_1_2021-06-01_2021-06-30.csv'),
 PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/KlineData/klinePremium/ETHUSDT/interval=1/ETHUSDT_premium_index_kline_1_2021-07-01_2021-07-31.csv'),
 PosixPath('/Users/damensavvasavvi/desktop/tmpMarketData/KlineData/klinePremium/ETHUSDT/interval=1/ETHUSDT_premium_index_kline_1_2021-08-01_2021-08-31.csv'),
 PosixPath('/Users/damensavvasavvi/desktop/tmpMarket

In [39]:
# Get instrument specs from Bybit API

def get_bybit_linear_instrument_info(symbol: str, testnet: bool = False) -> dict:
    base_url = "https://api-testnet.bybit.com" if testnet else "https://api.bybit.com"

    params = {
        "category": "linear",
        "symbol": symbol.upper(),
    }

    r = requests.get(
        f"{base_url}/v5/market/instruments-info",
        params=params,
        timeout=20,
    )
    r.raise_for_status()

    payload = r.json()

    if payload["retCode"] != 0:
        raise RuntimeError(payload)

    instruments = payload["result"]["list"]

    if not instruments:
        raise ValueError(f"No Bybit linear instrument found for {symbol}")

    return instruments[0]

info = get_bybit_linear_instrument_info(SYMBOL)

In [40]:
symbol = info["symbol"]              
base_coin = info["baseCoin"]         
quote_coin = info["quoteCoin"]      
settle_coin = info["settleCoin"]     

tick_size = info["priceFilter"]["tickSize"]
qty_step = info["lotSizeFilter"]["qtyStep"]

price_precision = int(info["priceScale"])
size_precision = abs(Decimal(qty_step).as_tuple().exponent)

min_qty = info["lotSizeFilter"]["minOrderQty"]
max_qty = info["lotSizeFilter"]["maxOrderQty"]
min_notional = info["lotSizeFilter"]["minNotionalValue"]

min_price = info["priceFilter"]["minPrice"]
max_price = info["priceFilter"]["maxPrice"]

max_leverage = Decimal(info["leverageFilter"]["maxLeverage"])
margin_init = Decimal("1") / max_leverage

In [41]:
CRYPTOPERP_INSTRUMENT = CryptoPerpetual(
    instrument_id=InstrumentId.from_str(f"{symbol}-LINEAR.BYBIT"),
    raw_symbol=Symbol(symbol),

    base_currency=Currency.from_str(base_coin),
    quote_currency=Currency.from_str(quote_coin),
    settlement_currency=Currency.from_str(settle_coin),

    is_inverse=False,

    price_precision=price_precision,
    size_precision=size_precision,

    price_increment=Price.from_str(tick_size),
    size_increment=Quantity.from_str(qty_step),

    multiplier=Quantity.from_str("1"),
    lot_size=Quantity.from_str("1"),

    min_quantity=Quantity.from_str(min_qty),
    max_quantity=Quantity.from_str(max_qty),

    min_notional=Money.from_str(f"{min_notional} {quote_coin}"),
    max_notional=None,

    min_price=Price.from_str(min_price),
    max_price=Price.from_str(max_price),

    margin_init=margin_init,
    margin_maint=Decimal("0"),

    maker_fee=Decimal("0.0002"),
    taker_fee=Decimal("0.00055"),

    ts_event=0,
    ts_init=0,

    info=info,
)

CRYPTOPERP_INSTRUMENT

CryptoPerpetual(id=ETHUSDT-LINEAR.BYBIT, raw_symbol=ETHUSDT, asset_class=CRYPTOCURRENCY, instrument_class=SWAP, quote_currency=USDT, is_inverse=False, price_precision=2, price_increment=0.01, size_precision=2, size_increment=0.01, multiplier=1, lot_size=1, margin_init=0.01, margin_maint=0, maker_fee=0.0002, taker_fee=0.00055, info={'symbol': 'ETHUSDT', 'contractType': 'LinearPerpetual', 'status': 'Trading', 'baseCoin': 'ETH', 'quoteCoin': 'USDT', 'launchTime': '1615766400000', 'deliveryTime': '0', 'deliveryFeeRate': '', 'priceScale': '2', 'leverageFilter': {'minLeverage': '1', 'maxLeverage': '100.00', 'leverageStep': '0.01'}, 'priceFilter': {'minPrice': '0.01', 'maxPrice': '199999.98', 'tickSize': '0.01'}, 'lotSizeFilter': {'maxOrderQty': '10000.00', 'minOrderQty': '0.01', 'qtyStep': '0.01', 'postOnlyMaxOrderQty': '10000.00', 'maxMktOrderQty': '2000.00', 'minNotionalValue': '5'}, 'unifiedMarginTrade': True, 'fundingInterval': 480, 'settleCoin': 'USDT', 'copyTrading': 'both', 'upperFund

In [46]:
df = pd.read_csv(raw_files[0])
df

,dataset,symbol,interval,start_time_ms,start_time_utc,open,high,low,close,volume,turnover
0,premium_index_kline,ETHUSDT,1,1615766400000,2021-03-15T00:00:00+00:00,0.002310,0.002310,0.002175,0.002175,NaN,NaN
1,premium_index_kline,ETHUSDT,1,1615766460000,2021-03-15T00:01:00+00:00,0.002175,0.002175,0.001314,0.001314,NaN,NaN
2,premium_index_kline,ETHUSDT,1,1615766520000,2021-03-15T00:02:00+00:00,0.001314,0.001314,0.001276,0.001276,NaN,NaN
3,premium_index_kline,ETHUSDT,1,1615766580000,2021-03-15T00:03:00+00:00,0.001276,0.001276,0.001150,0.001150,NaN,NaN
4,premium_index_kline,ETHUSDT,1,1615766640000,2021-03-15T00:04:00+00:00,0.001150,0.001447,0.001150,0.001447,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
24475,premium_index_kline,ETHUSDT,1,1617234900000,2021-03-31T23:55:00+00:00,0.001330,0.001608,0.001330,0.001608,NaN,NaN
24476,premium_index_kline,ETHUSDT,1,1617234960000,2021-03-31T23:56:00+00:00,0.001608,0.001608,0.001467,0.001467,NaN,NaN
24477,premium_index_kline,ETHUSDT,1,1617235020000,2021-03-31T23:57:00+00:00,0.001467,0.001601,0.001467,0.001601,NaN,NaN
24478,premium_index_kline,ETHUSDT,1,1617235080000,2021-03-31T23:58:00+00:00,0.001601,0.001601,0.001445,0.001445,NaN,NaN


In [ ]:
def process_kline_bars(
    csv_file: Path | str, 
    instrument: Instrument,
    premium_kline: bool = False
) -> pd.DataFrame:
    """
    Processes a CSV file into a validated DataFrame and Nautilus Bar objects 
    without creating intermediate DataFrame copies.
    """
    
    BAR_COLUMNS = ["open", "high", "low", "close", "volume"]

    # ---------------------------------------------------------------------------
    # 1. Load CSV
    # ---------------------------------------------------------------------------
    df = pd.read_csv(
        csv_file,
        engine="pyarrow",
    )

    required_columns = {
        "symbol",
        "interval",
        "start_time_ms",
        *BAR_COLUMNS,
    }

    missing_columns = required_columns.difference(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    if df.empty:
        raise ValueError(f"CSV file is empty: {csv_file}")


    # ---------------------------------------------------------------------------
    # 2. Validate symbol and interval
    # ---------------------------------------------------------------------------
    if df["symbol"].nunique() != 1:
        raise ValueError(
            f"CSV contains multiple symbols: {df['symbol'].unique().tolist()}"
        )

    intervals = pd.to_numeric(
        df["interval"],
        errors="raise",
    ).astype("int64")

    if intervals.nunique() != 1:
        raise ValueError(
            f"CSV contains multiple intervals: {intervals.unique().tolist()}"
        )

    interval_minutes = int(intervals.iloc[0])

    if not 1 <= interval_minutes < 60:
        raise ValueError(
            f"This script expects a minute interval below 60, got {interval_minutes}"
        )


    # ---------------------------------------------------------------------------
    # 3. Prepare the DataFrame (In-Place / No Copies)
    # ---------------------------------------------------------------------------
    
    # Drop any columns that are not required for the final output
    columns_to_keep = set(["start_time_ms"] + BAR_COLUMNS)
    columns_to_drop = [col for col in df.columns if col not in columns_to_keep]
    df.drop(columns=columns_to_drop, inplace=True)

    # Convert Bybit candle start time to candle close time directly into a new column
    df["timestamp"] = (
        pd.to_datetime(
            df["start_time_ms"],
            unit="ms",
            utc=True,
        )
        + pd.to_timedelta(interval_minutes, unit="m")
    )
    
    # Drop the old start_time_ms column as it is no longer needed
    df.drop(columns=["start_time_ms"], inplace=True)

    for column in BAR_COLUMNS:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )
    # Premium kline has no volume
    if premium_kline:
        df["volume"] = df["volume"].fillna(0.0)
    # Execute cleaning operations sequentially in-place to avoid copying memory
    df.dropna(subset=["timestamp", "open", "high", "low", "close", "volume"], inplace=True)
    df.sort_values("timestamp", kind="stable", inplace=True)
    df.drop_duplicates(subset="timestamp", keep="last", inplace=True)
    df.reset_index(drop=True, inplace=True)

    if df.empty:
        raise ValueError(
            f"No valid bar rows remained after processing {csv_file}"
        )


    # ---------------------------------------------------------------------------
    # 4. Define the Nautilus bar type
    # ---------------------------------------------------------------------------
    bar_type = BarType.from_str(
        f"{instrument.id.value}-{interval_minutes}-MINUTE-LAST-EXTERNAL"
    )


    # ---------------------------------------------------------------------------
    # 5. Convert DataFrame -> PyO3 bars -> legacy Cython bars
    # ---------------------------------------------------------------------------
    wrangler = BarDataWranglerV2(
        bar_type=str(bar_type),
        price_precision=instrument.price_precision,
        size_precision=instrument.size_precision,
    )

    pyo3_bars = wrangler.from_pandas(
        df=df,
        ts_init_delta=0,
    )

    bars = Bar.from_pyo3_list(pyo3_bars)
    
    # The bars list is loaded in memory for Nautilus validation/usage, 
    # but we return the raw DataFrame as requested.
    return df, bars , bar_type

In [ ]:
catalog = ParquetDataCatalog(str(CATALOG_DIR))
catalog.write_data([CRYPTOPERP_INSTRUMENT])

for DataFile in raw_files:
    # Assuming your function returns the dataframe, the Nautilus bars, and the BarType
    df, bars, bar_type = process_kline_bars(DataFile, CRYPTOPERP_INSTRUMENT, premium_kline=PREMIUM_KLINE)
    
    # Write only the Nautilus Bar objects to the catalog
    catalog.write_data(bars)
    
    print(f"Successfully cataloged {DataFile}")

File /Users/damensavvasavvi/Desktop/NautilusTrader/project/nautilusPremiumCatalog/data/crypto_perpetual/ETHUSDT-LINEAR.BYBIT/1970-01-01T00-00-00-000000000Z_1970-01-01T00-00-00-000000000Z.parquet already exists, skipping write
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/KlineData/klinePremium/ETHUSDT/interval=1/ETHUSDT_premium_index_kline_1_2021-03-01_2021-03-31.csv
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/KlineData/klinePremium/ETHUSDT/interval=1/ETHUSDT_premium_index_kline_1_2021-04-01_2021-04-30.csv
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/KlineData/klinePremium/ETHUSDT/interval=1/ETHUSDT_premium_index_kline_1_2021-05-01_2021-05-31.csv
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/KlineData/klinePremium/ETHUSDT/interval=1/ETHUSDT_premium_index_kline_1_2021-06-01_2021-06-30.csv
Successfully cataloged /Users/damensavvasavvi/desktop/tmpMarketData/KlineData/klinePremium/ETHUSDT/interval=1/